# Contextual Emotion Distribution Vectorizer (CEDV)
## Mental Health Analysis using Social Media Data (TensorFlow 2.x / Keras)

This notebook implements an end-to-end continuous emotion distribution vectorizer that maps conversational text and social media paragraphs into an 8-dimensional normalized affective vector.

### 8-Class Affective Taxonomy:
1. `depression` (`r/depression`)
2. `anxiety` (`r/anxiety`, `r/socialanxiety`, `r/PanicAttack`)
3. `stress_burnout` (`r/stress`, `r/burnout`, `r/offmychest`)
4. `adhd` (`r/adhd`, `r/adhdwomen`)
5. `autism` (`r/autism`, `r/aspergers`)
6. `loneliness` (`r/lonely`, `r/Needafriend`)
7. `anger` (`r/anger`, `r/vent`)
8. `positive_neutral` (`r/happy`, `r/CasualConversation`, `r/MadeMeSmile`, `r/Gratitude`)

---
### Dual Distribution Normalizations:
- **Option A (Simplex Distribution)**: Softmax with temperature $\tau$ ($\sum v_i = 1.0, v_i \ge 0$).
- **Option B (Z-Score Normalization)**: Relative emotional deviation ($\mu = 0.0, \sigma = 1.0$).


In [ ]:
# 1. Environment & GPU Verification
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running on Google Colab instance.")
    get_ipython().system("pip install -q pandas numpy scikit-learn requests rich seaborn matplotlib")
else:
    print("Running in local environment.")

import tensorflow as tf
import keras
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Accelerated GPU available: {gpus}")
    try:
        keras.mixed_precision.set_global_policy('mixed_float16')
        print("FP16 Mixed Precision active for optimal GPU throughput.")
    except Exception as e:
        print(f"Mixed precision notice: {e}")
else:
    print("No GPU detected; running on CPU fallback.")


In [ ]:
# 2. Imports and Configuration
import re
import html
import json
import time
import hashlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import keras
from keras import layers

CLASS_LABELS = [
    "depression",
    "anxiety",
    "stress_burnout",
    "adhd",
    "autism",
    "loneliness",
    "anger",
    "positive_neutral"
]
LABEL_TO_IDX = {l: i for i, l in enumerate(CLASS_LABELS)}
IDX_TO_LABEL = {i: l for i, l in enumerate(CLASS_LABELS)}
NUM_CLASSES = len(CLASS_LABELS)

print(f"Initialized CEDV Taxonomy with {NUM_CLASSES} classes: {CLASS_LABELS}")


In [ ]:
# 3. Data Ingestion & Preprocessing
DATASET_PATH = "data/dataset.csv"

if os.path.exists(DATASET_PATH):
    print(f"Loading balanced dataset from {DATASET_PATH}...")
    df = pd.read_csv(DATASET_PATH)
else:
    print("Dataset not found locally; scraping fresh keyless dataset...")
    from src.scraper import CEDVScraper
    scraper = CEDVScraper(request_delay=0.25)
    df = scraper.scrape_all(samples_per_class=550, output_csv=DATASET_PATH)

print(f"Dataset loaded: {len(df)} samples across {df['label'].nunique()} classes.")
print("
Class Distribution:")
print(df["label"].value_counts())


In [ ]:
# 4. Text Cleaning and Dataset Partitioning
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\br/([a-zA-Z0-9_]+)", r"\1", text)
    text = re.sub(r"\bu/([a-zA-Z0-9_-]+)", "", text)
    text = re.sub(r"\[(deleted|removed)\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"[\*_~`>#]", " ", text)
    text = re.sub(r"([!?.,])\1{3,}", r"\1\1\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply cleaning
df["clean_text"] = df["text"].astype(str).apply(clean_text)
df = df[df["clean_text"].apply(lambda t: len(t.split())) >= 3].copy()
df["class_id"] = df["label"].map(LABEL_TO_IDX).astype(int)
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

# Stratified Split (70% train, 15% val, 15% test)
train_val_df, test_df = train_test_split(df, test_size=0.15, stratify=df["class_id"], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1765, stratify=train_val_df["class_id"], random_state=42)

print(f"Split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Build tf.data datasets
BATCH_SIZE = 32

def make_ds(texts, labels, shuffle=False):
    x = np.array(texts, dtype=object)
    y = keras.utils.to_categorical(labels, num_classes=NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(len(x))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df["clean_text"].values, train_df["class_id"].values, shuffle=True)
val_ds = make_ds(val_df["clean_text"].values, val_df["class_id"].values)
test_ds = make_ds(test_df["clean_text"].values, test_df["class_id"].values)


In [ ]:
# 5. Build CEDV Architecture with GloVe 50d Warm-Start & Mask-Aware Dual Pooling
VOCAB_SIZE = 30000
SEQ_LENGTH = 128
EMBEDDING_DIM = 50
GLOVE_PATH = "data/glove.6B.50d.txt"

# Auto-download GloVe 6B 50d if running in fresh environment
if not os.path.exists(GLOVE_PATH):
    print("Downloading GloVe 6B 50d vectors (163 MB)...")
    os.makedirs("data", exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        "https://huggingface.co/kcz358/glove/resolve/main/glove.6B/glove.6B.50d.txt",
        GLOVE_PATH
    )
    print("GloVe download complete.")

@keras.saving.register_keras_serializable(package="cedv")
class MaskedFeaturePooling(layers.Layer):
    """Mask-aware pooling that eliminates padding artifacts for short text inputs."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs):
        features, token_ids = inputs
        mask = keras.ops.cast(keras.ops.not_equal(token_ids, 0), features.dtype)
        mask_exp = keras.ops.expand_dims(mask, -1)
        # Masked max: set padding to -1e4 so it never registers as maximum
        neg_pad = -1e4 * (1.0 - mask_exp)
        pooled_max = keras.ops.maximum(keras.ops.max(features + neg_pad, axis=1), 0.0)
        # Masked average: sum real tokens and divide by active token count
        lengths = keras.ops.maximum(keras.ops.sum(mask_exp, axis=1), 1.0)
        pooled_mean = keras.ops.sum(features * mask_exp, axis=1) / lengths
        return keras.ops.concatenate([pooled_max, pooled_mean], axis=-1)

    def compute_output_shape(self, input_shape):
        feat_shape, _ = input_shape
        return (feat_shape[0], feat_shape[2] * 2)

    def get_config(self):
        return super().get_config()

# Adapt TextVectorization
vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQ_LENGTH,
    name="text_vectorization"
)
vectorizer.adapt(train_df["clean_text"].values)
vocab = vectorizer.get_vocabulary()
actual_vocab_size = len(vocab)
print(f"Vocabulary size adapted: {actual_vocab_size} tokens")

# Align GloVe 50d Vectors
print(f"Aligning GloVe vectors from {GLOVE_PATH}...")
embedding_matrix = np.zeros((actual_vocab_size, EMBEDDING_DIM), dtype=np.float32)
glove_index = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        word = parts[0]
        vec = np.asarray(parts[1:], dtype=np.float32)
        if len(vec) == EMBEDDING_DIM:
            glove_index[word] = vec

hits, misses = 0, 0
for i, word in enumerate(vocab):
    if i == 0:
        continue
    vec = glove_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec
        hits += 1
    else:
        embedding_matrix[i] = np.random.normal(scale=0.05, size=(EMBEDDING_DIM,))
        misses += 1
print(f"GloVe Alignment: {hits} hits ({hits / actual_vocab_size * 100:.1f}%), {misses} OOV tokens")

# Full Architecture Definition with GloVe warm-start (trainable=True)
text_input = layers.Input(shape=(1,), dtype=tf.string, name="text_input")
token_ids = vectorizer(text_input)
x = layers.Embedding(
    input_dim=actual_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    trainable=True,
    name="embedding"
)(token_ids)
x = layers.SpatialDropout1D(0.25, name="spatial_dropout")(x)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm")(x)
x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu", name="conv1d")(x)
pooled = MaskedFeaturePooling(name="masked_pooling")([x, token_ids])
x = layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4), name="dense_proj")(pooled)
x = layers.Dropout(0.4, name="dropout")(x)
logits = layers.Dense(NUM_CLASSES, activation=None, dtype="float32", name="logits")(x)

cedv_model = keras.Model(inputs=text_input, outputs=logits, name="CEDV_Model")
cedv_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=0.05),
    metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")]
)
cedv_model.summary()


In [ ]:
# 6. Training CEDV Model
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

history = cedv_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# 7. Model Evaluation & Publication-Quality Visualizations
raw_logits = cedv_model.predict(test_ds)

# Simplex probabilities: Softmax
shifted = raw_logits - np.max(raw_logits, axis=-1, keepdims=True)
exp_v = np.exp(shifted)
pred_probs = exp_v / np.sum(exp_v, axis=-1, keepdims=True)

y_pred = np.argmax(pred_probs, axis=-1)
y_true = test_df["class_id"].values

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_LABELS, digits=4))

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Normalized Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=axes[0])
axes[0].set_title("Normalized Confusion Matrix (Test Set)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicted State", fontsize=11)
axes[0].set_ylabel("True State", fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

# Loss & Accuracy Trajectories
epochs_range = range(1, len(history.history["loss"]) + 1)
axes[1].plot(epochs_range, history.history["loss"], "b-o", label="Train Loss")
axes[1].plot(epochs_range, history.history["val_loss"], "r--s", label="Val Loss")
axes[1].set_title("Training Loss Trajectory", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Loss", fontsize=11)
axes[1].legend()
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# 8. Interactive Affective State Vectorizer
def vectorize_emotion(text: str, temperature: float = 1.0):
    """Predict continuous emotion distribution vector for input text."""
    cleaned = clean_text(text)
    input_t = tf.constant([cleaned], dtype=tf.string)
    logits = cedv_model(input_t, training=False).numpy()[0]
    
    # Simplex (Option A)
    scaled = logits / max(temperature, 1e-4)
    shifted = scaled - np.max(scaled)
    simplex = np.exp(shifted) / np.sum(np.exp(shifted))
    
    # Z-Score (Option B)
    mean_val = np.mean(logits)
    std_val = np.std(logits)
    zscore = (logits - mean_val) / (std_val + 1e-7)
    
    # Plot bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
    axes[0].barh(CLASS_LABELS, simplex * 100, color=colors)
    axes[0].set_xlabel("Simplex Probability (%)", fontsize=11)
    axes[0].set_title(f"Emotion Distribution (Primary: {CLASS_LABELS[np.argmax(simplex)]})", fontweight="bold")
    axes[0].grid(True, linestyle=":", alpha=0.6)
    
    axes[1].barh(CLASS_LABELS, zscore, color=colors)
    axes[1].axvline(0, color='black', linestyle='--', alpha=0.7)
    axes[1].axvline(0.5, color='orange', linestyle=':', label='+0.5σ threshold')
    axes[1].set_xlabel("Standard Deviations (Z-score σ)", fontsize=11)
    axes[1].set_title("Relative Affective Deviation", fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    plt.show()
    
    return {
        "text": cleaned,
        "primary_emotion": CLASS_LABELS[np.argmax(simplex)],
        "simplex_vector": simplex.round(4).tolist(),
        "zscore_vector": zscore.round(4).tolist()
    }

# Test with conversational query:
sample_query = "I am feeling so good today!"
res = vectorize_emotion(sample_query)
print(f"Input: '{res['text']}' -> Primary: {res['primary_emotion']}")
print("8D Continuous Simplex Vector:", res["simplex_vector"])
